In [1]:
import os
import cv2

def load_dataset_opencv(dataset_path):
    images = []
    labels = []

    # Scorre tutte le sottocartelle (una per lettera)
    for label in sorted(os.listdir(dataset_path)):
        subfolder = os.path.join(dataset_path, label)
        
        # Salta file che non sono cartelle
        if not os.path.isdir(subfolder):
            continue
        
        # Scorre tutte le immagini dentro ogni cartella
        for filename in os.listdir(subfolder):
            file_path = os.path.join(subfolder, filename)
            
            # Carica immagine in BGR (formato OpenCV)
            img = cv2.imread(file_path)
            if img is None:
                continue  # ignora file non validi
            
            images.append(img)
            labels.append(label)  # usa la sottocartella come etichetta
    
    return images, labels
train_path = "C:/Users/nicol/Desktop/Progetti/Machine learning/project/asl_alphabet" # metti qui il percorso vero
X_train, y_train = load_dataset_opencv(train_path)

print("Immagini train:", len(X_train))
print("Etichette train:", len(y_train))
print("Esempio dimensioni:", X_train[0].shape)

Immagini train: 2515
Etichette train: 2515
Esempio dimensioni: (400, 400, 3)


In [2]:
import cv2
import numpy as np
import os
import shutil

# --- Percorsi ---
dataset_path = "C:/Users/nicol/Desktop/Progetti/Machine learning/project/asl_alphabet"
output_path = "C:/Users/nicol/Desktop/Progetti/Machine learning/project/asl_alphabet_mask_cropped"

# Pulisce la cartella di output
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path, exist_ok=True)

# --- Funzione maschera dai pixel non neri ---
def create_mask_from_non_black(img_bgr, threshold=10):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mask = np.where(gray > threshold, 255, 0).astype(np.uint8)
    return mask

# --- Elaborazione ---
letters = sorted(os.listdir(dataset_path))
for letter in letters:
    folder = os.path.join(dataset_path, letter)
    if not os.path.isdir(folder):
        continue

    out_folder = os.path.join(output_path, letter)
    os.makedirs(out_folder, exist_ok=True)

    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Crea la maschera
        mask = create_mask_from_non_black(img)

        # Bounding box della mano
        ys, xs = np.where(mask > 0)
        if len(xs) == 0 or len(ys) == 0:
            continue

        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()

        # Crop della maschera
        crop_mask = mask[y0:y1+1, x0:x1+1]

        # Salva la maschera croppata
        out_mask_path = os.path.join(out_folder, filename)
        cv2.imwrite(out_mask_path, crop_mask)


In [ ]:
import os
import cv2
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from math import cos, sin, radians
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1) FUNZIONE PER ESTRARRE IL CONTORNO
# ============================================================

def extract_contour(mask):
    """
    Ritorna una lista di coordinate (y,x) dei pixel di contorno:
    - pixel bianchi che hanno almeno un vicino nero
    - oppure pixel bianchi sul bordo dell'immagine
    """

    h, w = mask.shape
    contour = []

    # scorre tutti i pixel bianchi
    for y in range(h):
        for x in range(w):
            if mask[y, x] == 0:
                continue

            # Caso 1: bordo immagine
            if x == 0 or x == w-1 or y == 0 or y == h-1:
                contour.append((y, x))
                continue

            # Caso 2: pixel bianco con vicino nero
            neighborhood = mask[y-1:y+2, x-1:x+2]
            if np.any(neighborhood == 0):
                contour.append((y, x))

    return contour


# ============================================================
# 2) FEATURE: DISTANZE RADIALI (360 DIREZIONI)
# ============================================================

def compute_radial_signature(mask, contour, num_angles=360):
    """
    Calcola 360 distanze dal centro al primo pixel di contorno
    lungo ciascuna direzione.
    """

    h, w = mask.shape

    # centro dell'immagine
    cy = h // 2
    cx = w // 2

    # converti contorno in set per velocità
    contour_set = set(contour)

    signature = []

    for angle in range(num_angles):
        theta = radians(angle)
        dx = cos(theta)
        dy = sin(theta)

        dist = 0
        max_steps = max(h, w)

        found = False
        for step in range(1, max_steps):
            x = int(cx + dx * step)
            y = int(cy + dy * step)

            # fuori immagine → distanza massima
            if x < 0 or x >= w or y < 0 or y >= h:
                signature.append(step)
                found = True
                break

            # pixel di contorno trovato
            if (y, x) in contour_set:
                signature.append(step)
                found = True
                break

        if not found:
            signature.append(max_steps)

    return np.array(signature, dtype=np.float32)


# ============================================================
# 3) CARICA TUTTE LE MASCHERE E GENERA LE FEATURE
# ============================================================

dataset_path = "C:/Users/nicol/Desktop/Progetti/Machine learning/project/asl_alphabet_mask_cropped"

X = []
y = []

for label in sorted(os.listdir(dataset_path)):
    folder = os.path.join(dataset_path, label)
    if not os.path.isdir(folder):
        continue

    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        mask = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue

        # estrai contorno
        contour = extract_contour(mask)
        if len(contour) == 0:
            continue

        # calcola feature
        signature = compute_radial_signature(mask, contour, num_angles=360)

        X.append(signature)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Campioni totali:", len(X))


# ============================================================
# 4) SPLIT TRAIN / VALIDATION / TEST
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", len(X_train))
print("Valid:", len(X_valid))
print("Test:", len(X_test))


# ============================================================
# 5) RICERCA DEL MIGLIOR K SU VALIDATION
# ============================================================

best_k = None
best_acc = -1

for k in range(1, 26):
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    pred = knn.predict(X_valid)
    acc = accuracy_score(y_valid, pred)

    print(f"k={k} -> Validation accuracy = {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_k = k

print("\nMiglior k trovato:", best_k)
print("Accuracy valida:", best_acc)


# ============================================================
# 6) ALLENA MODELLO FINALE SU TRAIN + VALID
# ============================================================

X_final_train = np.vstack((X_train, X_valid))
y_final_train = np.concatenate((y_train, y_valid))

knn_final = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean")
knn_final.fit(X_final_train, y_final_train)

# ============================================================
# 7) TEST FINALE
# ============================================================

y_pred = knn_final.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print("\n===============================")
print("ACCURACY FINALE SU TEST:", final_accuracy)
print("===============================")


Campioni totali: 2515
Train: 1760
Valid: 377
Test: 378
k=1 -> Validation accuracy = 0.8541
k=2 -> Validation accuracy = 0.8064
k=3 -> Validation accuracy = 0.7931
k=4 -> Validation accuracy = 0.7798
k=5 -> Validation accuracy = 0.7507
k=6 -> Validation accuracy = 0.7188
k=7 -> Validation accuracy = 0.7109
k=8 -> Validation accuracy = 0.6950
k=9 -> Validation accuracy = 0.6923
k=10 -> Validation accuracy = 0.6790
k=11 -> Validation accuracy = 0.6790
k=12 -> Validation accuracy = 0.6684
k=13 -> Validation accuracy = 0.6790
k=14 -> Validation accuracy = 0.6631
k=15 -> Validation accuracy = 0.6472
k=16 -> Validation accuracy = 0.6419
k=17 -> Validation accuracy = 0.6446
k=18 -> Validation accuracy = 0.6393
k=19 -> Validation accuracy = 0.6260
k=20 -> Validation accuracy = 0.6154
k=21 -> Validation accuracy = 0.6180
k=22 -> Validation accuracy = 0.6048
k=23 -> Validation accuracy = 0.5915
k=24 -> Validation accuracy = 0.6021
k=25 -> Validation accuracy = 0.5836

Miglior k trovato: 1
Accurac